# Chapter 7: Search In-Depth
## Topic 6: MMR — Maximal Marginal Relevance

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every retrieval method built in Topics 1-5 (BM25, dense, hybrid via RRF) optimizes for one thing only: how relevant is each individual document to the query, considered completely independently of every other document in the result set.
- This creates a specific, common failure: the top results can all be near-duplicates of each other, saying essentially the same thing in slightly different wording, while genuinely distinct but slightly-less-relevant information never makes it into the final result set at all.
- Concretely: a query about a penalty rule might return three separate chunks that all restate the exact same 1% penalty fact (from an FAQ, a policy document, and a procedure document), while a chunk about a genuinely useful exception to that rule never appears, simply because it individually scores a bit lower than the third near-duplicate.
- MMR's core idea: don't just rank by relevance to the query — rank by relevance to the query **minus** redundancy with whatever has already been selected. Build the result set one document at a time, and at each step pick whichever remaining candidate offers the best trade-off between "how relevant is this" and "how different is this from what I've already picked."
- This is a re-ranking / re-selection step applied *after* an initial retrieval pass — MMR doesn't find new candidates, it re-orders and re-selects from a candidate pool that's already been retrieved.
- Why "marginal": this is an economics-flavored term meaning the *additional* value something adds given what's already there, not its value in isolation. A document's raw relevance score is fixed, but its marginal value changes at every step of building the result set, because it depends on what's already been selected.


### 2. Internal Working — Step by Step

**The core formula, in plain terms:**

- At each step, for every remaining candidate document, compute: (a weight, lambda) times (its relevance to the query) minus (1 minus that weight) times (its highest similarity to anything already selected).
- Pick whichever candidate scores highest on this combined formula, add it to the result set, and repeat until enough documents have been selected.

**Step by step:**

1. Start with a candidate pool — the output of an initial retrieval pass (like the top-15 or top-20 from hybrid retrieval in Topic 4).
2. Compute each candidate's similarity to the query — this is just the retrieval score already available.
3. Start with an empty "selected" list.
4. Repeat until enough documents are selected: for every candidate not yet selected, compute its combined score (relevance term minus redundancy term), where the redundancy term is the highest similarity between this candidate and anything already in the selected list (zero if nothing has been selected yet). Pick whichever candidate scores highest, add it to selected, and remove it from the candidate pool.
5. Return the selected list as the final, diversity-aware result set.

**Why this actually produces diverse results:**

- On the very first pick, since nothing has been selected yet, the formula reduces to pure relevance ranking — the single most relevant document always wins the first slot.
- From the second pick onward, a document that's highly relevant to the query but also highly similar to something already selected gets its score dragged down by the redundancy penalty. A document that's somewhat less relevant in isolation, but genuinely different from what's already chosen, can now win instead.
- This is the whole mechanism: redundant documents get penalized in proportion to how similar they are to what's already chosen, letting a moderately-relevant-but-distinct document earn a slot instead of a third near-duplicate.

**The role of the lambda weight:**

- Setting lambda to 1 removes diversity consideration entirely — this becomes standard top-k relevance-only retrieval.
- Setting lambda to 0 ignores the query's relevance entirely after the first pick, and just tries to make every subsequent pick as different as possible from what's already chosen, regardless of whether it's actually relevant.
- A typical practical range balances relevance as the primary driver with diversity as a meaningful secondary correction.
- There's no universally correct value — it depends on how redundant the underlying corpus actually is, and how much value genuinely diverse-but-secondary information provides for the specific use case.


### 3. How This Is Implemented in This Project

- MMR fits into the pipeline after hybrid retrieval (Topic 4) produces a ranked candidate pool, and re-ranks that pool down to a smaller final set that actually gets passed to the generation layer — explicitly balancing relevance against redundancy.
- This is a cheap, local re-ranking step: no new retrieval calls, no new embeddings beyond what the candidate pool already has computed.
- Why this specifically matters for this kind of knowledge base: real structural redundancy exists across source documents — the same fact can be stated in an FAQ, restated with more formal framing in a policy reference document, and restated again as a procedural step in an SOP document. Without MMR, a single query could easily return several chunks that are functionally the same fact stated three different ways, wasting valuable context slots on repeated information instead of surfacing genuinely complementary details.
- For the similarity function used to check relevance to the query, the retrieval score already computed during hybrid retrieval can be reused directly. For the similarity function used to check redundancy between two candidate documents, dense embeddings must be used specifically — this is a semantic comparison between two pieces of text, not a query-document match, and a keyword-matching score like BM25 isn't designed for this kind of document-to-document comparison. Since chunks are typically already embedded from the retrieval step, these same embeddings can be reused for the redundancy check with no additional embedding calls needed.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Computational cost:** MMR's cost grows with the number of documents to select multiplied by the size of the candidate pool, since at each selection step it has to scan through all remaining candidates. For a moderate candidate pool (a few dozen candidates, selecting a handful of final documents), this is a tiny number of simple similarity computations on already-computed embeddings — effectively instant. At much larger candidate pools (hundreds or thousands of candidates), this becomes a real cost, which is exactly why production systems first narrow down to a moderate candidate pool through retrieval before ever applying MMR, rather than running MMR over the entire corpus.
- **The lambda tuning problem:** exactly analogous to tuning BM25's k1/b parameters or RRF's k constant — there's no universally correct value, and it requires either a labeled evaluation set or a qualitative review of actual outputs to tune sensibly for a specific corpus. Unlike some of those other parameters, lambda has a fairly direct, interpretable effect, making it worth doing a quick qualitative pass — literally reading the actual top results at a few different lambda values for real queries — before committing to a production default.
- **Debugging a bad MMR result:** if MMR excludes an obviously-correct document from the final result set, first check whether that document had high similarity to an earlier, higher-relevance selection — this might actually be MMR working exactly as intended (penalizing genuine redundancy), which could instead indicate the candidate pool itself lacked real diversity, or that lambda is set too low for this particular query type. If MMR still returns near-duplicates despite being applied, a common bug is accidentally using the query-relevance similarity function for the document-to-document redundancy check, or computing similarity in the wrong space entirely (like comparing keyword-based scores instead of dense embeddings).
- **Monitoring:** track the average pairwise similarity within each returned result set over time — a rising trend suggests either the lambda weight needs adjustment, or the candidate pool itself has grown more redundant (for example, after a new document was ingested that duplicates existing content). Also track how often MMR's final top-1 pick differs from the raw relevance-ranked top-1 pick — this only happens when the selected set already contains something very similar to that top pick, so tracking this frequency reveals how often redundancy is actually a live, real problem for the query distribution being handled.
- **Latency:** negligible at reasonable candidate-pool sizes, since MMR reuses already-computed embeddings and adds no new model inference calls — only fast in-memory similarity computations.
- **Deployment:** MMR is a pure post-processing function — no new infrastructure, no new service, no database changes required. It slots directly after the retrieval call and before the final chunks are handed off to whatever comes next (like a generation prompt).


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **MMR vs simply deduplicating near-identical chunks:** a simpler alternative is to just detect and remove chunks that are above some similarity threshold to already-selected chunks — a binary duplicate-removal approach rather than MMR's continuous relevance-diversity trade-off. MMR is strictly more expressive, since it can choose a moderately redundant-but-highly-relevant document over a completely-novel-but-barely-relevant one, which a simple binary threshold cannot represent. For a corpus with mostly distinct documents and only occasional near-duplicates, simple threshold-based deduplication may capture most of the benefit with far less tuning complexity — worth evaluating both before committing to the more complex option.
- **Where in the pipeline MMR should run relative to reranking (Topic 7):** MMR re-orders for diversity, while a more accurate relevance-scoring reranker re-orders for relevance accuracy — these solve different problems, and the right order matters. A common, correct pattern: retrieve candidates first, then rerank them for more accurate relevance scores, then apply MMR on that now-more-accurately-scored pool to select a final diverse set. Running MMR before a more accurate reranking step means MMR's relevance signal is based on the less accurate initial retrieval score rather than the more accurate reranked score — generally the wrong order.
- **Fixed result-set size vs adaptive size:** a fixed-size final result set is simple, but MMR naturally supports stopping early if all remaining candidates have very low marginal value (relevance minus redundancy penalty falling below some threshold), avoiding forcing in a genuinely low-value final document just to hit a fixed count. This becomes worth considering once there's a proper way to budget how much context downstream consumption can actually use — a genuinely low-value final chunk consumes that budget without adding proportional value.


### 6. Alternatives and When to Use Each

- **MMR (this topic's approach):** best for corpora with real content redundancy across multiple source documents — for example, the same fact restated across an FAQ, a policy document, and a procedure document. Use when relevance-only ranking is *measurably* returning near-duplicate results, verified by actually inspecting real query outputs, not just assumed.
- **Simple similarity-threshold deduplication:** best for corpora where redundancy is rare and mostly exact-or-near-exact, especially if duplicate detection is already partially handled earlier in the pipeline (like at ingestion time). Use when MMR's continuous relevance-diversity trade-off is more complexity than the actual redundancy pattern in the corpus actually warrants.
- **Clustering-based diversity (cluster candidates, then pick one representative per cluster):** best for much larger candidate pools where MMR's cost becomes a real concern, or where the corpus has clearly separable topic clusters. Use when guaranteed coverage across distinct clusters matters more than MMR's greedy, order-dependent selection process.
- **No diversity handling at all (relevance-only ranking):** best for corpora with genuinely low redundancy, or use cases where the single most relevant answer is what actually matters and secondary or complementary information has low value. Use when a qualitative review of actual retrieval outputs shows redundancy isn't really a problem in practice — don't add MMR's tuning burden without evidence that it's actually needed.


### 7. Common Mistakes and Production Failures

- Applying MMR before actually checking whether redundancy is a real, measured problem — MMR adds a tunable parameter that needs justification from real evidence, not blind adoption because it sounds like a good idea.
- Using the wrong similarity function for the redundancy check — this must be a genuine document-to-document semantic comparison using dense embeddings, not accidentally reusing a query-relevance score or a keyword-based score that was never designed for this kind of comparison.
- Setting the diversity weight too aggressively without realizing it can hurt precision — at an extreme diversity setting, MMR essentially prioritizes being different over being relevant, which can pull in genuinely off-topic candidates purely because they happen to be dissimilar to what's already selected. Always sanity-check outputs qualitatively when tuning toward more diversity.
- Running MMR before a more accurate relevance-reranking step instead of after — this means MMR's relevance judgments are based on a less accurate initial score rather than the more refined one that a proper reranking step would provide.
- Applying MMR over an enormous, unfiltered candidate pool instead of first narrowing down with retrieval — this defeats the performance benefit of MMR being a cheap, local re-ranking step and turns it into a real computational cost.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What specific problem does MMR solve that plain relevance-based ranking cannot?
  A: Plain relevance ranking can return several near-duplicate top results that all say essentially the same thing, because each document is scored independently of what else is in the result set. MMR explicitly penalizes redundancy with already-selected documents, allowing genuinely distinct but slightly-less-relevant information to also make it into the final result set.

- Q: What does the lambda parameter control in MMR?
  A: It balances relevance against diversity. A value of 1 makes MMR behave like plain relevance ranking with no diversity consideration; a value of 0 makes it prioritize being different from what's already selected almost regardless of relevance; typical practical values sit in between, treating relevance as the primary driver with diversity as a secondary correction.

**Intermediate**

- Q: Why must MMR's redundancy check use dense embeddings rather than the retrieval method's original relevance score?
  A: The redundancy check is a document-to-document semantic comparison — how similar is this candidate to something already selected — which is a fundamentally different comparison than query-to-document relevance. A keyword-based relevance score isn't designed for comparing two documents to each other; dense embeddings, which represent meaning as vectors, are the right tool for this kind of similarity check.

- Q: Why does MMR always pick the single most relevant document first, regardless of the lambda value?
  A: Because the redundancy term depends on similarity to already-selected documents, and on the very first pick, nothing has been selected yet — so that term is zero for every candidate, and the formula reduces to pure relevance ranking for that first pick only.

**Advanced**

- Q: A colleague suggests applying MMR before a more accurate relevance-reranking step instead of after. What's wrong with this ordering?
  A: MMR's relevance term relies on whatever relevance score is available at the time it runs. If MMR runs before the more accurate reranking step, it's making its diversity trade-off decisions based on the less accurate initial retrieval score rather than the more refined one. The correct order is to rerank for accurate relevance first, then apply MMR on that more accurately-scored pool to make the final diversity-aware selection.

- Q: How would you decide between full MMR and simple similarity-threshold deduplication for a given corpus?
  A: Inspect actual retrieval outputs for real queries. If the corpus mostly returns clearly distinct documents with only rare exact-or-near-exact duplicates, simple threshold-based deduplication likely captures most of the benefit with far less tuning complexity. If the corpus has genuine structural redundancy — the same information restated multiple times across different documents with real but partial relevance differences — MMR's continuous trade-off is worth the added complexity, since simple deduplication can't represent "moderately redundant but highly relevant" versus "completely novel but barely relevant" the way MMR's formula can.

**Scenario-based**

- Q: Monitoring shows the average pairwise similarity within your top-5 results has been rising over the past month. Walk through your diagnosis.
  A: First check whether new content was recently added to the knowledge base that duplicates existing information — this would organically increase redundancy in the candidate pool being fed to MMR. If the candidate pool composition hasn't meaningfully changed, consider whether the lambda weight needs adjusting toward more diversity. Also spot-check a sample of actual queries and their returned results to confirm the metric reflects a real user-facing problem, not just noise in a handful of edge-case queries.

**Follow-up questions to expect:**

- "How would you measure whether MMR is actually improving downstream answer quality, not just diversifying results for its own sake?"
- "What happens to MMR's behavior if the candidate pool itself is too small or too homogeneous?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Greedy, order-dependent selection:** MMR builds its result set one document at a time, and each choice depends on everything already chosen — this means the algorithm makes locally-good decisions at each step, but doesn't guarantee the globally best possible combination of documents. This greedy nature is a known, accepted trade-off for the speed and simplicity it provides.
- **The relevance-diversity trade-off is a general pattern, not unique to retrieval:** the same underlying tension — optimizing for individual quality versus optimizing for a good combination as a whole — shows up in many other contexts, like recommendation systems trying to avoid showing a user five nearly-identical items in a row.
- **MMR's dependence on the quality of the initial candidate pool:** MMR can only select from what it's given — if the initial retrieval pass already failed to surface a genuinely relevant and distinct document, no amount of diversity-aware re-ranking can recover it. This reinforces why MMR is a re-ranking step on top of good retrieval, not a substitute for it.

### 10. Quick Revision Summary (for last-minute interview prep)

> MMR fixes a blind spot that every purely relevance-based retrieval method shares: individually scoring documents against a query, with no awareness of redundancy across the result set, can produce a top-k list full of near-duplicates while genuinely useful complementary information gets excluded. MMR builds its result set one document at a time, at each step picking whichever remaining candidate offers the best combined score of relevance to the query minus similarity to what's already been selected, controlled by a tunable weight (lambda) that balances the two. The first pick is always purely the most relevant document, since nothing has been selected yet to be redundant with. This is a cheap, local re-ranking step applied after retrieval — it needs a genuine document-to-document similarity check (dense embeddings) for its redundancy term, should run after (not before) any more accurate relevance-reranking step, and should only be adopted once redundancy is confirmed as a real, measured problem in actual retrieval outputs — not applied by default just because it sounds beneficial.


### Module 1: Setup — A Candidate Pool With Real Redundancy

Build a candidate pool where some documents are near-duplicates of each other (restating the same fact), using offline TF-IDF+SVD dense vectors for both query-relevance and document-to-document similarity.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- candidate pool with real redundancy
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 3 of these chunks restate the SAME penalty fact in different words
# (FAQ style, policy style, SOP style) -- deliberate redundancy.
# The other 2 are genuinely different, secondary information.
CANDIDATE_POOL = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",          # 0: FAQ style
    "Fixed Deposit premature closure attracts a penalty charge of 1 percent as per RBI guidelines.",    # 1: Policy style, same fact
    "Step 3 of the withdrawal SOP: apply a 1 percent penalty deduction on the FD interest rate.",         # 2: SOP style, same fact again
    "The premature withdrawal penalty is waived if the FD is closed due to the death of the depositor.", # 3: genuinely different (nominee exception)
    "Senior citizens receive an additional 0.5 percent interest rate on all Fixed Deposit tenures.",     # 4: genuinely different topic
]

QUERY = "what is the penalty for premature FD withdrawal"

def normalize_vector(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

# build dense vectors for query-relevance AND document-to-document
# similarity -- same embedding space, reused for both purposes
vectorizer = TfidfVectorizer()
sparse_matrix = vectorizer.fit_transform(CANDIDATE_POOL)
svd = TruncatedSVD(n_components=4, random_state=42)
doc_vectors = svd.fit_transform(sparse_matrix)
doc_vectors_norm = np.array([normalize_vector(v) for v in doc_vectors])

query_vector = normalize_vector(svd.transform(vectorizer.transform([QUERY]))[0])

# relevance of each candidate to the query -- this is the retrieval score
relevance_scores = np.array([cosine_similarity(query_vector, v) for v in doc_vectors_norm])

print("=" * 70)
print("CANDIDATE POOL AND RELEVANCE TO QUERY")
print("=" * 70)
print(f"Query: '{QUERY}'\n")
for i, (text, score) in enumerate(zip(CANDIDATE_POOL, relevance_scores)):
    print(f"  Doc {i} | relevance={score:.4f} | {text[:65]}...")

plain_top3 = np.argsort(relevance_scores)[::-1][:3]
print(f"\nPlain relevance-only top-3 (no diversity awareness): {list(plain_top3)}")
print("Notice: Docs 0, 1, 2 are very likely to be the top-3, since they")
print("all restate the SAME fact and score similarly high -- Doc 3's")
print("genuinely different, useful nominee-exception information gets")
print("excluded purely because it scores a bit lower in isolation.")
print("\nModule 1 complete. Run Module 2 next.")


CANDIDATE POOL AND RELEVANCE TO QUERY
Query: 'what is the penalty for premature FD withdrawal'

  Doc 0 | relevance=0.6506 | Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Doc 1 | relevance=0.2038 | Fixed Deposit premature closure attracts a penalty charge of 1 pe...
  Doc 2 | relevance=0.7180 | Step 3 of the withdrawal SOP: apply a 1 percent penalty deduction...
  Doc 3 | relevance=0.9343 | The premature withdrawal penalty is waived if the FD is closed du...
  Doc 4 | relevance=0.0012 | Senior citizens receive an additional 0.5 percent interest rate o...

Plain relevance-only top-3 (no diversity awareness): [np.int64(3), np.int64(2), np.int64(0)]
Notice: Docs 0, 1, 2 are very likely to be the top-3, since they
all restate the SAME fact and score similarly high -- Doc 3's
genuinely different, useful nominee-exception information gets
excluded purely because it scores a bit lower in isolation.

Module 1 complete. Run Module 2 next.


### Module 2: The MMR Algorithm

Built from scratch, applied to the candidate pool from Module 1.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: The MMR algorithm
#
# MMR score(d) = lambda * relevance(d, query)
#                - (1 - lambda) * max_similarity(d, already_selected)
# ------------------------------------------------------------------

def mmr_select(candidate_indices: list, relevance_scores: np.ndarray,
               doc_vectors: np.ndarray, k: int, lambda_weight: float = 0.6) -> list:
    """
    Greedily selects k documents from candidate_indices, balancing
    relevance to the query against redundancy with already-selected docs.
    """
    remaining = list(candidate_indices)
    selected = []

    while remaining and len(selected) < k:
        best_score = -float("inf")
        best_doc = None

        for doc_id in remaining:
            relevance_term = relevance_scores[doc_id]

            if selected:
                # redundancy penalty: highest similarity to anything already picked
                redundancy_term = max(
                    cosine_similarity(doc_vectors[doc_id], doc_vectors[s]) for s in selected
                )
            else:
                redundancy_term = 0.0  # nothing selected yet -- no penalty possible

            mmr_score = lambda_weight * relevance_term - (1 - lambda_weight) * redundancy_term

            if mmr_score > best_score:
                best_score = mmr_score
                best_doc = doc_id

        selected.append(best_doc)
        remaining.remove(best_doc)

    return selected


all_indices = list(range(len(CANDIDATE_POOL)))
mmr_result = mmr_select(all_indices, relevance_scores, doc_vectors_norm, k=3, lambda_weight=0.6)

print("=" * 70)
print("MMR SELECTION (lambda=0.6, k=3)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")
for rank, doc_id in enumerate(mmr_result, start=1):
    print(f"  MMR pick {rank}: Doc {doc_id} | relevance={relevance_scores[doc_id]:.4f} | {CANDIDATE_POOL[doc_id][:60]}...")

print(f"\nPlain relevance-only top-3 was: {list(plain_top3)}")
print(f"MMR top-3 is:                   {mmr_result}")
if set(mmr_result) != set(plain_top3):
    print("\nMMR selected a DIFFERENT set -- it likely swapped out one of the")
    print("redundant penalty-restatement docs for the genuinely distinct")
    print("nominee-exception document (Doc 3), because that document's lower")
    print("relevance score was outweighed by its low redundancy with picks")
    print("already made.")
else:
    print("\nMMR selected the SAME set as plain relevance ranking this time --")
    print("this can happen if lambda is high enough that relevance still")
    print("dominates, or if the redundant docs are not similar enough to")
    print("each other to trigger a meaningful penalty. Try Module 3 to see")
    print("how changing lambda affects this.")
print("\nModule 2 complete. Run Module 3 next.")


MMR SELECTION (lambda=0.6, k=3)
Query: 'what is the penalty for premature FD withdrawal'

  MMR pick 1: Doc 3 | relevance=0.9343 | The premature withdrawal penalty is waived if the FD is clos...
  MMR pick 2: Doc 0 | relevance=0.6506 | Premature withdrawal of FD incurs a 1 percent penalty on the...
  MMR pick 3: Doc 2 | relevance=0.7180 | Step 3 of the withdrawal SOP: apply a 1 percent penalty dedu...

Plain relevance-only top-3 was: [np.int64(3), np.int64(2), np.int64(0)]
MMR top-3 is:                   [3, 0, 2]

MMR selected the SAME set as plain relevance ranking this time --
this can happen if lambda is high enough that relevance still
dominates, or if the redundant docs are not similar enough to
each other to trigger a meaningful penalty. Try Module 3 to see
how changing lambda affects this.

Module 2 complete. Run Module 3 next.


### Module 3: The Effect of Lambda

Runs MMR at several lambda values on the same candidate pool, showing the shift from pure relevance to pure diversity.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Effect of lambda on the final selection
# ------------------------------------------------------------------

print("=" * 70)
print("MMR RESULTS ACROSS DIFFERENT LAMBDA VALUES (k=3)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")

for lambda_weight in [1.0, 0.7, 0.5, 0.3, 0.0]:
    result = mmr_select(all_indices, relevance_scores, doc_vectors_norm, k=3, lambda_weight=lambda_weight)
    labels = [f"Doc{d}" for d in result]
    print(f"  lambda={lambda_weight:.1f} -> {labels}")

print("\nAt lambda=1.0: pure relevance ranking, MMR degenerates to plain top-k.")
print("As lambda decreases: redundant documents get pushed out in favor of")
print("documents that are more different from what is already selected,")
print("even if those documents are individually less relevant to the query.")
print("At lambda=0.0: relevance is ignored entirely after the first pick --")
print("purely maximizing difference from prior picks, which can pull in")
print("something only weakly related to the query at all.")
print("\nThis is the concrete, measurable version of the theory's claim:")
print("there is no single 'correct' lambda -- it must be chosen based on")
print("how much the corpus actually needs a diversity correction, tuned")
print("by qualitatively reviewing real outputs like the ones printed above.")
print("\nModule 3 complete. All Topic 6 modules done.")

print()
print("=" * 70)
print("CHAPTER 7 TOPIC 6 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  MMR re-ranks an ALREADY-RETRIEVED candidate pool -- it does not
  retrieve new candidates.

  MMR_score(d) = lambda * relevance(d, query)
                 - (1 - lambda) * max_similarity(d, already_selected)

  The FIRST pick is always the most relevant document -- the redundancy
  term is zero until something has been selected.

  lambda=1 -> pure relevance (no diversity). lambda=0 -> pure diversity
  (ignores relevance after the first pick). Typical range: 0.5-0.7.

  Redundancy check MUST use document-to-document dense similarity,
  never a query-relevance score or keyword-based score.

  Only adopt MMR after confirming redundancy is a REAL, measured
  problem in actual retrieval outputs -- not by default.
""")


MMR RESULTS ACROSS DIFFERENT LAMBDA VALUES (k=3)
Query: 'what is the penalty for premature FD withdrawal'

  lambda=1.0 -> ['Doc3', 'Doc2', 'Doc0']
  lambda=0.7 -> ['Doc3', 'Doc2', 'Doc0']
  lambda=0.5 -> ['Doc3', 'Doc0', 'Doc1']
  lambda=0.3 -> ['Doc3', 'Doc1', 'Doc0']
  lambda=0.0 -> ['Doc0', 'Doc1', 'Doc4']

At lambda=1.0: pure relevance ranking, MMR degenerates to plain top-k.
As lambda decreases: redundant documents get pushed out in favor of
documents that are more different from what is already selected,
even if those documents are individually less relevant to the query.
At lambda=0.0: relevance is ignored entirely after the first pick --
purely maximizing difference from prior picks, which can pull in
something only weakly related to the query at all.

This is the concrete, measurable version of the theory's claim:
there is no single 'correct' lambda -- it must be chosen based on
how much the corpus actually needs a diversity correction, tuned
by qualitatively reviewing real o